In [1]:
from pymatgen.core import Structure
import crystal_toolkit

fe_zno_fe = Structure.from_file("fe_zno_fe_final1.cif")
fe_zno_fe

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from __future__ import annotations
import warnings
from dataclasses import dataclass, field
from itertools import product
from typing import List, Optional, Tuple
import numpy as np
from pymatgen.analysis.interfaces.zsl import ZSLGenerator
from pymatgen.core import Structure
from pymatgen.core.surface import SlabGenerator, get_symmetrically_distinct_miller_indices
 
DEFAULT_WEIGHTS = {"strain": 2.0, "area": 0.01, "broken_bonds": 0.5, "polarity": 5.0}

In [4]:
from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

API_KEY = "pPxBVAcW8uMoR8udpokQXBGFSpiEX39d"

with MPRester(API_KEY) as mpr:
    fe  = mpr.get_structure_by_material_id("mp-13")
    mgo = mpr.get_structure_by_material_id("mp-1265")
    zno = mpr.get_structure_by_material_id("mp-2229")

# Convert to standard cells (VERY IMPORTANT)
fe  = SpacegroupAnalyzer(fe).get_conventional_standard_structure()
mgo = SpacegroupAnalyzer(mgo).get_conventional_standard_structure()
zno = SpacegroupAnalyzer(zno).get_conventional_standard_structure()

# Add oxidation states (fixes your warnings)
fe.add_oxidation_state_by_element({"Fe": 0})
mgo.add_oxidation_state_by_element({"Mg": 2, "O": -2})
zno.add_oxidation_state_by_element({"Zn": 2, "O": -2})

In [5]:
def get_top_species(structure, tol=0.2):
    z_coords = [site.coords[2] for site in structure]
    z_max = max(z_coords)

    top_sites = [site for site in structure
        if abs(site.coords[2] - z_max) < tol]

    species = [site.specie.symbol for site in top_sites]
    return max(set(species), key=species.count)


def pick_termination(cib, film, required_top=None):
    best_term = cib.terminations[0]
    best_score = -1

    for term in cib.terminations:
        try:
            interfaces = list(cib.get_interfaces(
                termination=term,
                substrate_thickness=2,
                film_thickness=2,
                vacuum_over_film=0,
                in_layers=True,
            ))

            if not interfaces:
                continue

            struct = interfaces[0]
            top_species = get_top_species(struct)

            # ENFORCE symmetry constraint
            if required_top is not None:
                if top_species != required_top:
                    continue

            # scoring (same as yours)
            score = 0
            if top_species == "O":
                score += 2
            if top_species in ["Zn", "Mg"]:
                score += 1

            if score > best_score:
                best_score = score
                best_term = term

        except Exception:
            continue

    return best_term

In [6]:
def get_matches(substrate, film, max_miller=2):
    analyzer = SubstrateAnalyzer(
        film_max_miller=max_miller,
        substrate_max_miller=max_miller,
    )

    matches = list(analyzer.calculate(film=film, substrate=substrate))

    # remove duplicates
    seen, unique = set(), []
    for m in matches:
        key = (m.substrate_miller, m.film_miller)
        if key not in seen:
            seen.add(key)
            unique.append(m)

    return unique

def choose_consistent_match(matches, substrate, film, ref_a, ref_b):
    best = matches[0]
    best_error = float("inf")

    for m in matches:
        try:
            cib = CoherentInterfaceBuilder(
                substrate_structure=substrate,
                film_structure=film,
                substrate_miller=m.substrate_miller,
                film_miller=m.film_miller,
            )

            zsl = list(cib.zsl_matches)[0]
            sl = np.array(zsl.substrate_sl_vectors)

            a = np.linalg.norm(sl[0])
            b = np.linalg.norm(sl[1])

            error = abs(a - ref_a) + abs(b - ref_b)

            if error < best_error:
                best_error = error
                best = m

        except Exception:
            continue

    return best

In [7]:
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer

def find_interfaces(substrate, film, max_miller=2):
    analyzer = SubstrateAnalyzer(
        film_max_miller=max_miller,
        substrate_max_miller=max_miller,
    )
    matches = list(analyzer.calculate(film=film, substrate=substrate))
    # deduplicate — same hkl pair appearing multiple times is just
    # different supercell sizes of the same orientation relationship
    seen = set()
    unique = []
    for m in matches:
        key = (m.substrate_miller, m.film_miller)
        if key not in seen:
            seen.add(key)
            unique.append(m)
    return unique[:5]

def best_scored_match(matches, w_strain=2.0, w_area=0.01):
    return min(matches, key=lambda m: 
        w_strain * m.von_mises_strain + w_area * m.match_area)

In [8]:
def run_multilayer_interface_search(structures):
    results = {}

    for i in range(len(structures) - 1):
        sub = structures[i]
        film = structures[i + 1]

        print(f"\nInterface {i}: {sub.formula} | {film.formula}")

        matches = find_interfaces(sub, film)

        if not matches:
            print("No good matches found")
        else:
            for j, m in enumerate(matches):
                print(f"  Match {j+1}:")
                print(  "strain:", m.von_mises_strain)
                print(  "sub_miller:", m.substrate_miller)
                print(  "film_miller:", m.film_miller)
                print(   "area:", m.match_area)

        results[i] = matches

    return results

structures = [fe, mgo, zno, mgo, fe]

results = run_multilayer_interface_search(structures)


Interface 0: Fe2 | Mg4 O4
  Match 1:
strain: 0.01617481086232934
sub_miller: (1, 0, 0)
film_miller: (1, 0, 0)
area: 70.35863745416937
  Match 2:
strain: 0.018585681130303844
sub_miller: (1, 1, 0)
film_miller: (1, 0, 0)
area: 211.0759123625081
  Match 3:
strain: 0.022652431918430565
sub_miller: (1, 1, 1)
film_miller: (1, 0, 0)
area: 211.07591236250812
  Match 4:
strain: 0.014506897020307587
sub_miller: (2, 1, 0)
film_miller: (1, 0, 0)
area: 105.53795618125406
  Match 5:
strain: 0.023640028036704005
sub_miller: (2, 1, 1)
film_miller: (1, 0, 0)
area: 316.6138685437621

Interface 1: Mg4 O4 | Zn4 O4
  Match 1:
strain: 0.017041269104661343
sub_miller: (1, 0, 0)
film_miller: (1, 0, 0)
area: 150.607314923648
  Match 2:
strain: 0.017916101288103913
sub_miller: (1, 1, 0)
film_miller: (1, 0, 0)
area: 338.86645857820804
  Match 3:
strain: 0.010756772542288913
sub_miller: (1, 1, 1)
film_miller: (1, 0, 0)
area: 338.86645857820804
  Match 4:
strain: 0.015342811039209316
sub_miller: (2, 1, 0)
film_mi

In [9]:
def check_symmetric_terminations(bilayers, structures):
    warnings = []

    # Example: MgO–ZnO and ZnO–MgO
    if len(bilayers) >= 3:
        left = bilayers[1]
        right = bilayers[2]

        left_top = get_top_species(left)
        right_top = get_top_species(right)

        if left_top != right_top:
            warnings.append(
                f"Asymmetric MgO terminations: {left_top} vs {right_top}"
            )

    for w in warnings:
        print("WARNING:", w)

    return warnings

In [10]:
def build_interface(substrate, film, match, layers_sub, layers_film, required_top=None):

    cib = CoherentInterfaceBuilder(
        substrate_structure=substrate,
        film_structure=film,
        substrate_miller=match.substrate_miller,
        film_miller=match.film_miller,
    )

    termination = pick_termination(cib, film, required_top)

    interfaces = list(cib.get_interfaces(
        termination=termination,
        substrate_thickness=layers_sub,
        film_thickness=layers_film,
        vacuum_over_film=0,
        gap=2.0,
        in_layers=True,
    ))

    struct = min(interfaces, key=lambda x: x.num_sites)
    top_species = get_top_species(struct)

    return struct, top_species

In [11]:
def get_matches(substrate, film, max_miller=2):
    analyzer = SubstrateAnalyzer(
        film_max_miller=max_miller,
        substrate_max_miller=max_miller,
    )
    matches = list(analyzer.calculate(film=film, substrate=substrate))

    # remove duplicates (same hkl pair)
    seen, unique = set(), []
    for m in matches:
        key = (m.substrate_miller, m.film_miller)
        if key not in seen:
            seen.add(key)
            unique.append(m)

    return unique  # sorted by strain already

In [12]:
def choose_consistent_match(matches, substrate, film, ref_a, ref_b):
    best = matches[0]
    best_error = float("inf")

    for m in matches:
        try:
            cib = CoherentInterfaceBuilder(
                substrate_structure=substrate,
                film_structure=film,
                substrate_miller=m.substrate_miller,
                film_miller=m.film_miller,
            )

            zsl = list(cib.zsl_matches)[0]
            sl = np.array(zsl.substrate_sl_vectors)

            a = np.linalg.norm(sl[0])
            b = np.linalg.norm(sl[1])

            error = abs(a - ref_a) + abs(b - ref_b)

            if error < best_error:
                best_error = error
                best = m

        except Exception:
            continue

    return best

In [13]:
def build_interface(substrate, film, match, layers_sub, layers_film):
    cib = CoherentInterfaceBuilder(
        substrate_structure=substrate,
        film_structure=film,
        substrate_miller=match.substrate_miller,
        film_miller=match.film_miller,
    )

    interfaces = list(cib.get_interfaces(
        termination=cib.terminations[0],
        substrate_thickness=layers_sub,
        film_thickness=layers_film,
        vacuum_over_film=0,
        gap=2.0,
        in_layers=True,
    ))

    return min(interfaces, key=lambda x: x.num_sites)

In [14]:
# Enforce same in-plane lattice (critical)

def match_lattice_to_reference(structure, ref_a, ref_b):
    a_ratio = ref_a / structure.lattice.a
    b_ratio = ref_b / structure.lattice.b

    na = int(round(a_ratio))
    nb = int(round(b_ratio))

    if na != 1 or nb != 1:
        structure = structure.copy()
        structure.make_supercell([na, nb, 1])

    return structure

In [15]:
def assemble_layers(bilayers, ref_lattice):
    species = []
    cart_coords = []

    # add first interface completely
    for site in bilayers[0]:
        species.append(site.specie)
        cart_coords.append(site.coords)

    current_z = max(c[2] for c in cart_coords)

    # add only film parts for next layers
    for bl in bilayers[1:]:
        film_sites = [bl[i] for i in bl.film_indices]

        min_z = min(s.coords[2] for s in film_sites)
        shift = current_z + 1.75 - min_z

        for site in film_sites:
            species.append(site.specie)
            new_coord = site.coords.copy()
            new_coord[2] += shift
            cart_coords.append(new_coord)

        current_z = max(c[2] for c in cart_coords)

    # build final lattice (same in-plane)
    final_lattice = Lattice.from_parameters(
        a=ref_lattice.a,
        b=ref_lattice.b,
        c=current_z + 1.75,
        alpha=ref_lattice.alpha,
        beta=ref_lattice.beta,
        gamma=ref_lattice.gamma,
    )

    frac = [final_lattice.get_fractional_coords(c) for c in cart_coords]

    return Structure(final_lattice, species, frac)

In [16]:
def build_full_heterostructure(fe, mgo, zno):
    materials = [fe, mgo, zno, mgo, fe]
    layer_config = [(4,2), (2,2), (2,2), (2,4)]

    print("Step 1: Finding matches...")
    all_matches = [
        get_matches(materials[i], materials[i+1])
        for i in range(len(materials)-1)
    ]
    
    # Step 2 — Anchor interface (Fe|MgO)
    print("Step 2: Building anchor interface...")
    anchor_match = best_scored_match(all_matches[0])

    anchor = build_interface(
        fe, mgo, anchor_match,
        layers_sub=4, layers_film=2
    )

    ref_a, ref_b = anchor.lattice.a, anchor.lattice.b
    print(f"Reference lattice: a={ref_a:.3f}, b={ref_b:.3f}")

    bilayers = [anchor]

    # Step 3 — Build rest consistently
    
    print("Step 3: Building remaining interfaces...")

    for i in range(1, 4):
        sub = materials[i]
        film = materials[i+1]

        match = choose_consistent_match(all_matches[i], sub, film,
            ref_a, ref_b)

        bl = build_interface(sub, film, match,
            *layer_config[i])

        bl = match_lattice_to_reference(bl, ref_a, ref_b)

        bilayers.append(bl)

        print(f"Interface {i}: {bl.num_sites} atoms")

    # Step 4 — Assemble final structure
    print("Step 4: Assembling structure...")
    final = assemble_layers(bilayers, anchor.lattice)

    print(f"Final structure: {final.num_sites} atoms")
    print(final)

    return final

In [17]:
from mp_api.client import MPRester
if __name__ == "__main__":
    from pymatgen.core import Structure, Lattice
    from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

with MPRester("pPxBVAcW8uMoR8udpokQXBGFSpiEX39d") as mpr:
    fe   = mpr.get_structure_by_material_id("mp-13") 
    mgo  = mpr.get_structure_by_material_id("mp-1265")
    zno  = mpr.get_structure_by_material_id("mp-2229")

    fe = SpacegroupAnalyzer(fe).get_conventional_standard_structure()
    mgo = SpacegroupAnalyzer(mgo).get_conventional_standard_structure()
    zno = SpacegroupAnalyzer(zno).get_conventional_standard_structure()

    # oxidation states (important)
    fe.add_oxidation_state_by_element({"Fe": 0})
    mgo.add_oxidation_state_by_element({"Mg": 2, "O": -2})
    zno.add_oxidation_state_by_element({"Zn": 2, "O": -2})

    final = build_full_heterostructure(fe, mgo, zno)
    print("\nSaved: Fe_MgO_ZnO_MgO_Fe.cif")

Retrieving MaterialsDoc documents: 100%|███████| 1/1 [00:00<00:00, 13107.20it/s]


Step 1: Finding matches...
Step 2: Building anchor interface...
Reference lattice: a=8.589, b=8.589
Step 3: Building remaining interfaces...
Interface 1: 136 atoms
Interface 2: 136 atoms
Interface 3: 136 atoms
Step 4: Assembling structure...
Final structure: 344 atoms
Full Formula (Mg68 Zn32 Fe144 O100)
Reduced Formula: Mg17Zn8Fe36O25
abc   :   8.589106   8.589106  47.497878
angles:  90.000000  90.000000  90.000000
pbc   :       True       True       True
Sites (344)
  #  SP           a          b          c
---  ----  --------  ---------  ---------
  0  Mg2+  0.25       0.75       0.283885
  1  Mg2+  0.5       -0          0.283885
  2  Mg2+  0.25       0.25       0.283885
  3  Mg2+  0.5        0.5        0.283885
  4  Mg2+  0.75       0.75       0.283885
  5  Mg2+  0         -0          0.283885
  6  Mg2+  0.75       0.25       0.283885
  7  Mg2+  0          0.5        0.283885
  8  Mg2+  0.25       0.5        0.239735
  9  Mg2+  0.5        0.75       0.239735
 10  Mg2+  0.25      -0 

In [18]:
final

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/pymatgen/analysis/local_env.py:4021: UserWarning:

No oxidation states specified on sites! For better results, set the site oxidation states in the structure.

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/pymatgen/analysis/local_env.py:4023: UserWarning:

No oxidation states specified on sites! For better results, set the site oxidation states in the structure.

/home/akanksha/Siesta_5/miniconda3/envs/matsci-ct311/lib/python3.11/site-packages/pymatgen/analysis/local_env.py:3935: UserWarning:

CrystalNN: cannot locate an appropriate radius, covalent or atomic radii will be used, this can lead to non-optimal results.



# 2nd try

In [19]:
from __future__ import annotations
import warnings
import numpy as np
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder, Interface
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer
from pymatgen.core import Structure, Lattice
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
 
warnings.filterwarnings("ignore")

In [20]:
def get_unique_matches(substrate, film, max_miller=2):
    """SubstrateAnalyzer matches, deduplicated by (hkl_sub, hkl_film)."""
    analyzer = SubstrateAnalyzer(
        film_max_miller=max_miller,
        substrate_max_miller=max_miller,
    )
    matches = list(analyzer.calculate(film=film, substrate=substrate))
    seen, unique = set(), []
    for m in matches:
        key = (m.substrate_miller, m.film_miller)
        if key not in seen:
            seen.add(key)
            unique.append(m)
    return unique

def best_scored_match(matches, w_strain=2.0, w_area=0.005):
    """Pick match minimising w_strain*strain + w_area*area (Pareto trade-off)."""
    return min(matches, key=lambda m: w_strain * m.von_mises_strain
                                     + w_area  * m.match_area)

In [21]:
def pick_match_for_cell(matches, ref_a, ref_b, tol=3.0):
    """From a list of matches, return the one whose CIB in-plane cell
    is closest to (ref_a, ref_b). Falls back to best-strain if none within tol."""
    best_m, best_delta = matches[0], float("inf")
    for m in matches:
        try:
            cib = CoherentInterfaceBuilder(
                substrate_structure=m._substrate,
                film_structure=m._film,
                substrate_miller=m.substrate_miller,
                film_miller=m.film_miller,
            )
            sl = np.array(list(cib.zsl_matches)[0].substrate_sl_vectors)
            a, b = np.linalg.norm(sl[0]), np.linalg.norm(sl[1])
            delta = abs(a - ref_a) + abs(b - ref_b)
            if delta < best_delta:
                best_delta, best_m = delta, m
        except Exception:
            continue
    if best_delta > tol:
        print(f"    ⚠ Closest match is {best_delta:.2f} Å from ref — using best-strain.")
        return matches[0]
    return best_m

In [22]:
def build_bilayer(substrate, film, match, substrate_layers=4, film_layers=2):
    """Build one CIB Interface from a SubstrateAnalyzer match."""
    cib = CoherentInterfaceBuilder(
        substrate_structure=substrate,
        film_structure=film,
        substrate_miller=match.substrate_miller,
        film_miller=match.film_miller,
    )
    interfaces = list(cib.get_interfaces(
        termination=cib.terminations[0],
        gap=2.0,
        vacuum_over_film=0,
        substrate_thickness=substrate_layers,
        film_thickness=film_layers,
        in_layers=True,
    ))
    return min(interfaces, key=lambda x: x.num_sites)

In [23]:
def get_top_species(iface, indices) -> str:
    """Return element symbol of topmost atom among given site indices."""
    sites = [iface[i] for i in indices]
    top   = max(sites, key=lambda s: s.coords[2])
    return top.specie.element.symbol

def pick_termination(cib, substrate, film) -> tuple:
    """
    Choose termination so that substrate top ≠ film bottom in charge sign
    (avoids same-charge contact: O-O or cation-cation).
    Falls back to terminations[0] if none satisfy the criterion.
    """
    from pymatgen.core.periodic_table import Element
    anions = {'O', 'N', 'S', 'F', 'Cl'}
 
    for term in cib.terminations:
        try:
            ifaces = list(cib.get_interfaces(
                termination=term, gap=2.0,
                vacuum_over_film=0,
                substrate_thickness=1, film_thickness=1,
                in_layers=True,
            ))
            if not ifaces:
                continue
            iface      = ifaces[0]
            sub_top    = get_top_species(iface, iface.substrate_indices)
            film_bot   = get_top_species(iface, iface.film_indices)   # min z = bottom
            # flip: film bottom = lowest z film atom
            film_sites = [iface[i] for i in iface.film_indices]
            film_bot   = min(film_sites, key=lambda s: s.coords[2]).specie.element.symbol
 
            sub_is_anion  = sub_top  in anions
            film_is_anion = film_bot in anions
            # good contact: one anion, one cation
            if sub_is_anion != film_is_anion:
                return term
        except Exception:
            continue
    return cib.terminations[0]

In [24]:
def tile_to_ref(slab: Structure, ref_a: float, ref_b: float, tol=0.5) -> Structure:
    """
    If slab's a or b differs from reference by more than tol Å,
    tile it with make_supercell to match. Only integer multiples supported.
    """
    na = round(ref_a / slab.lattice.a)
    nb = round(ref_b / slab.lattice.b)
    if na != 1 or nb != 1:
        slab = slab.copy()
        slab.make_supercell([na, nb, 1])
    return slab

def assemble(bilayers: list[Interface], ref_a: float, ref_b: float) -> Structure:
    """
    Stack bilayers in Cartesian space. Film only from bilayer 1 onward.
    All film sites are placed using their Cartesian z-coordinate only —
    in-plane (x,y) positions are taken as-is from each Interface.
    The final lattice uses ref_a, ref_b in-plane.
    """
    species, cart = [], []
 
    # First bilayer in full
    for site in bilayers[0].sites:
        species.append(site.specie)
        cart.append(site.coords.copy())
    current_top = max(c[2] for c in cart)
 
    for bl in bilayers[1:]:
        film_sites = [bl[i] for i in bl.film_indices]
        film_z_min = min(s.coords[2] for s in film_sites)
        z_shift    = (current_top + 2.0) - film_z_min
 
        for site in film_sites:
            species.append(site.specie)
            c = site.coords.copy()
            c[2] += z_shift
            cart.append(c)
 
        current_top = max(c[2] for c in cart)
 
    ref = bilayers[0].lattice
    new_lat = Lattice.from_parameters(
        a=ref_a, b=ref_b, c=current_top + 2.0,
        alpha=ref.alpha, beta=ref.beta, gamma=ref.gamma,
    )
    frac = [new_lat.get_fractional_coords(c) for c in cart]
    return Structure(new_lat, species, frac, coords_are_cartesian=False)

In [25]:
def remove_overlaps(structure: Structure, cutoff=1.5) -> Structure:
    to_remove = set()
    for i in range(len(structure)):
        if i in to_remove:
            continue
        for j in range(i + 1, len(structure)):
            if j not in to_remove and structure.get_distance(i, j) < cutoff:
                to_remove.add(j)
    if to_remove:
        print(f"  Removing {len(to_remove)} overlapping atoms (d < {cutoff} Å)")
        structure.remove_sites(sorted(to_remove, reverse=True))
    return structure

def _check_termination_symmetry(bilayers):
    """For a symmetric stack A|B|C|B|A, interfaces 1 and 2 (both B|C and C|B)
    should present the same species at the contact plane.
    Prints a warning if they don't — does not abort, since DFT will relax."""
    if len(bilayers) < 3:
        return
    # Interface 1 (MgO->ZnO): what species is at the MgO top (substrate top)?
    bl1 = bilayers[1]
    bl2 = bilayers[2]
    sub1_top  = get_top_species(bl1, bl1.substrate_indices)
    sub2_top  = get_top_species(bl2, bl2.substrate_indices)
 
    film1_sites = [bl1[i] for i in bl1.film_indices]
    film2_sites = [bl2[i] for i in bl2.film_indices]
    film1_bot = min(film1_sites, key=lambda s: s.coords[2]).specie.element.symbol
    film2_bot = min(film2_sites, key=lambda s: s.coords[2]).specie.element.symbol
 
    print(f"  Interface 1 (MgO→ZnO): MgO top={sub1_top}, ZnO bottom={film1_bot}")
    print(f"  Interface 2 (ZnO→MgO): ZnO top={sub2_top}, MgO bottom={film2_bot}")
 
    if sub1_top == film2_bot and film1_bot == sub2_top:
        print("Terminations are symmetric")
    else:
        print("Asymmetric terminations — interfaces are not equivalent.")
        print("This is acceptable for pre-relaxation; DFT will resolve it.")

In [26]:
def build_heterostructure(fe, mgo, zno):
    structures  = [fe, mgo, zno, mgo, fe]
    pair_labels = ["Fe→MgO", "MgO→ZnO", "ZnO→MgO", "MgO→Fe"]
    layer_count = [(4,2), (2,2), (2,2), (2,4)]
 
    # Step 1: get all matches (one SubstrateAnalyzer call per unique pair)
    print("Step 1: Finding interface matches")
    unique_pairs = [(fe,mgo), (mgo,zno), (zno,mgo), (mgo,fe)]
    all_matches = []
    for (sub, film), label in zip(unique_pairs, pair_labels):
        matches = get_unique_matches(sub, film)
        all_matches.append(matches)
        print(f"  {label}: {len(matches)} unique (hkl) pairs, "
              f"best strain={matches[0].von_mises_strain:.4f}  "
              f"sub{matches[0].substrate_miller}‖film{matches[0].film_miller}")
 
    # Step 2: build anchor bilayer (Fe|MgO, best match)
    print("\nStep 2: Building anchor bilayer (Fe|MgO)")
    anchor_match = all_matches[0][0]
    anchor_bl    = build_bilayer(fe, mgo, anchor_match, 4, 2)
    ref_a, ref_b = anchor_bl.lattice.a, anchor_bl.lattice.b
    print(f"  Anchor cell: a={ref_a:.3f} b={ref_b:.3f} Å  ({anchor_bl.num_sites} sites)")
 
    # Step 3: pick best cell-consistent match for remaining pairs
    print("\nStep 3: Building remaining bilayers (cell-consistent)")
    bilayers = [anchor_bl]
    for idx in range(1, 4):
        label = pair_labels[idx]
        sub, film = unique_pairs[idx]
        sub_l, film_l = layer_count[idx]
        chosen = best_scored_match(all_matches[idx])
        bl = build_bilayer(sub, film, chosen, sub_l, film_l)
        bilayers.append(bl)
        da = abs(bl.lattice.a - ref_a)
        db = abs(bl.lattice.b - ref_b)
        print(f"  {label}: {bl.num_sites} sites | "
              f"a={bl.lattice.a:.3f}(Δ{da:.2f}) b={bl.lattice.b:.3f}(Δ{db:.2f}) Å")
 
    # Step 3b: check symmetric terminations
    print("\nStep 3b: Checking termination symmetry")
    _check_termination_symmetry(bilayers)
 
    # Step 4: assemble
    print("\nStep 4: Assembling structure")
    final = assemble(bilayers, ref_a, ref_b)
    print(f"  Raw: {final.num_sites} sites | c={final.lattice.c:.3f} Å")
 
    # Step 5: clean
    print("\nStep 5: Cleaning")
    final = remove_overlaps(final)
    final.sort()
    print(f"  Final: {final.num_sites} sites | {final.formula}")
    print(f"  Lattice: a={final.lattice.a:.3f} b={final.lattice.b:.3f} c={final.lattice.c:.3f} Å")
    return final

In [27]:
if __name__ == "__main__":
    Fe  = Structure.from_spacegroup('Im-3m', Lattice.cubic(2.87), ['Fe'], [[0,0,0]])
    MgO = Structure.from_spacegroup('Fm-3m', Lattice.cubic(4.211),
          ['Mg','O'], [[0,0,0],[0.5,0.5,0.5]])
    ZnO = Structure.from_spacegroup('P6_3mc', Lattice.hexagonal(3.2495, 5.2069),
          ['Zn','O'], [[1/3,2/3,0],[1/3,2/3,0.375]])
 
    fe  = SpacegroupAnalyzer(Fe).get_conventional_standard_structure()
    mgo = SpacegroupAnalyzer(MgO).get_conventional_standard_structure()
    zno = SpacegroupAnalyzer(ZnO).get_conventional_standard_structure()
    fe.add_oxidation_state_by_element({'Fe': 0})
    mgo.add_oxidation_state_by_element({'Mg': 2, 'O': -2})
    zno.add_oxidation_state_by_element({'Zn': 2, 'O': -2})
 
    final = build_heterostructure(fe, mgo, zno)
    final

Step 1: Finding interface matches
  Fe→MgO: 31 unique (hkl) pairs, best strain=0.0150  sub(1, 0, 0)‖film(1, 0, 0)
  MgO→ZnO: 47 unique (hkl) pairs, best strain=0.0177  sub(1, 0, 0)‖film(0, 0, 1)
  ZnO→MgO: 40 unique (hkl) pairs, best strain=0.0188  sub(0, 0, 1)‖film(1, 0, 0)
  MgO→Fe: 31 unique (hkl) pairs, best strain=0.0144  sub(1, 0, 0)‖film(1, 0, 0)

Step 2: Building anchor bilayer (Fe|MgO)
  Anchor cell: a=8.610 b=8.610 Å  (136 sites)

Step 3: Building remaining bilayers (cell-consistent)
  MgO→ZnO: 80 sites | a=8.422(Δ0.19) b=12.633(Δ4.02) Å
  ZnO→MgO: 80 sites | a=8.328(Δ0.28) b=12.275(Δ3.67) Å
  MgO→Fe: 88 sites | a=8.422(Δ0.19) b=8.422(Δ0.19) Å

Step 3b: Checking termination symmetry
  Interface 1 (MgO→ZnO): MgO top=Mg, ZnO bottom=O
  Interface 2 (ZnO→MgO): ZnO top=O, MgO bottom=O
Asymmetric terminations — interfaces are not equivalent.
This is acceptable for pre-relaxation; DFT will resolve it.

Step 4: Assembling structure
  Raw: 240 sites | c=35.267 Å

Step 5: Cleaning
  Re

In [28]:
final

# 3rd try

In [29]:
from __future__ import annotations
import numpy as np
import warnings
from pymatgen.analysis.interfaces.coherent_interfaces import CoherentInterfaceBuilder
from pymatgen.analysis.interfaces.substrate_analyzer import SubstrateAnalyzer
from pymatgen.core import Structure, Lattice
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

warnings.filterwarnings("ignore")

def get_matches(substrate, film, max_miller=2):
    analyzer = SubstrateAnalyzer(
        film_max_miller=max_miller,
        substrate_max_miller=max_miller,
    )
    matches = list(analyzer.calculate(film=film, substrate=substrate))

    seen, unique = set(), []
    for m in matches:
        key = (m.substrate_miller, m.film_miller)
        if key not in seen:
            seen.add(key)
            unique.append(m)
    return unique


def score_match(m, ref_a=None, ref_b=None):
    score = 2.0 * m.von_mises_strain + 0.005 * m.match_area

    # add lattice consistency penalty if reference exists
    if ref_a is not None:
        try:
            cib = CoherentInterfaceBuilder(
                substrate_structure=m._substrate,
                film_structure=m._film,
                substrate_miller=m.substrate_miller,
                film_miller=m.film_miller,
            )
            zsl = list(cib.zsl_matches)[0]
            sl = np.array(zsl.substrate_sl_vectors)

            a = np.linalg.norm(sl[0])
            b = np.linalg.norm(sl[1])

            score += 0.2 * (abs(a - ref_a) + abs(b - ref_b))
        except:
            pass
    return score


def best_match(matches, ref_a=None, ref_b=None):
    return min(matches, key=lambda m: score_match(m, ref_a, ref_b))

In [30]:
def pick_termination_fast(cib):
    # heuristic: prefer mixed ionic contact (O vs metal)
    for term in cib.terminations:
        name = str(term)
        if "O" in name:
            return term
    return cib.terminations[0]

In [31]:
def build_bilayer(substrate, film, match, layers_sub=2, layers_film=2):

    cib = CoherentInterfaceBuilder(
        substrate_structure=substrate,
        film_structure=film,
        substrate_miller=match.substrate_miller,
        film_miller=match.film_miller,
    )

    term = pick_termination_fast(cib)

    interfaces = list(cib.get_interfaces(
        termination=term,
        gap=0.5,  # small physical gap
        vacuum_over_film=0,
        substrate_thickness=layers_sub,
        film_thickness=layers_film,
        in_layers=True,
    ))

    return min(interfaces, key=lambda x: x.num_sites)

In [32]:
def rotate_to_ref(coords, src_lat, ref_lat):
    frac = src_lat.get_fractional_coords(coords)

    new = (
        frac[0] * ref_lat.matrix[0] +
        frac[1] * ref_lat.matrix[1]
    )

    new[2] = coords[2]
    return new

def tile_to_match(struct, ref_a, ref_b):
    na = round(ref_a / struct.lattice.a)
    nb = round(ref_b / struct.lattice.b)

    if na > 1 or nb > 1:
        struct = struct.copy()
        struct.make_supercell([na, nb, 1])

    return struct

In [33]:
def assemble(bilayers, ref_a, ref_b):

    ref_lat = bilayers[0].lattice
    species, coords = [], []

    # first layer
    for site in bilayers[0]:
        species.append(site.specie)
        coords.append(site.coords.copy())

    current_top = max(c[2] for c in coords)

    for bl in bilayers[1:]:

        film_sites = [bl[i] for i in bl.film_indices]

        rotated_sites = []

        # rotate first
        for site in film_sites:
            c = rotate_to_ref(site.coords, bl.lattice, ref_lat)
            rotated_sites.append((site.specie, c))

        # compute shift AFTER rotation
        min_z = min(c[2] for _, c in rotated_sites)
        shift = (current_top + 1.0) - min_z

        for sp, c in rotated_sites:
            c[2] += shift
            species.append(sp)
            coords.append(c)

        current_top = max(c[2] for c in coords)

    # build lattice
    final_lat = Lattice.from_parameters(
        a=ref_a,
        b=ref_b,
        c=current_top + 1.0,
        alpha=ref_lat.alpha,
        beta=ref_lat.beta,
        gamma=ref_lat.gamma,
    )

    frac = [final_lat.get_fractional_coords(c) for c in coords]

    return Structure(final_lat, species, frac)

In [34]:
def remove_overlaps(struct, cutoff=1.4):
    remove = set()

    for i in range(len(struct)):
        for j in range(i + 1, len(struct)):
            if struct.get_distance(i, j) < cutoff:
                remove.add(j)

    struct.remove_sites(sorted(remove, reverse=True))
    return struct

In [35]:
def build_full_heterostructure(fe, mgo, zno):

    print("Step 1: Finding matches...")

    pairs = [(fe, mgo), (mgo, zno), (zno, mgo), (mgo, fe)]
    all_matches = [get_matches(a, b) for a, b in pairs]

    print("Step 2: Anchor MgO | ZnO")

    anchor_match = best_match(all_matches[1])
    anchor = build_bilayer(mgo, zno, anchor_match)

    ref_a, ref_b = anchor.lattice.a, anchor.lattice.b

    print(f"Reference: a={ref_a:.3f}, b={ref_b:.3f}")

    bilayers = [anchor]

    print("Step 3: Building others")

    for i, (sub, film) in enumerate(pairs):
        if i == 1:
            continue

        match = best_match(all_matches[i], ref_a, ref_b)
        bl = build_bilayer(sub, film, match)

        # tile Fe layers if needed
        if sub.formula.startswith("Fe"):
            bl = tile_to_match(bl, ref_a, ref_b)

        bilayers.append(bl)

    print("Step 4: Assembling...")

    final = assemble(bilayers, ref_a, ref_b)

    print("Step 5: Cleaning overlaps...")
    final = remove_overlaps(final)

    print(f"Final atoms: {final.num_sites}")

    return final

In [36]:
if __name__ == "__main__":

    Fe  = Structure.from_spacegroup('Im-3m', Lattice.cubic(2.87), ['Fe'], [[0,0,0]])
    MgO = Structure.from_spacegroup('Fm-3m', Lattice.cubic(4.211),
          ['Mg','O'], [[0,0,0],[0.5,0.5,0.5]])
    ZnO = Structure.from_spacegroup('P6_3mc', Lattice.hexagonal(3.25, 5.2),
          ['Zn','O'], [[1/3,2/3,0],[1/3,2/3,0.375]])

    fe  = SpacegroupAnalyzer(Fe).get_conventional_standard_structure()
    mgo = SpacegroupAnalyzer(MgO).get_conventional_standard_structure()
    zno = SpacegroupAnalyzer(ZnO).get_conventional_standard_structure()

    fe.add_oxidation_state_by_element({'Fe': 0})
    mgo.add_oxidation_state_by_element({'Mg': 2, 'O': -2})
    zno.add_oxidation_state_by_element({'Zn': 2, 'O': -2})

    final_2 = build_full_heterostructure(fe, mgo, zno)

Step 1: Finding matches...
Step 2: Anchor MgO | ZnO
Reference: a=5.955, b=16.844
Step 3: Building others
Step 4: Assembling...
Step 5: Cleaning overlaps...
Final atoms: 277


In [37]:
final_2